In [6]:
from pathlib import Path
import sys

def _find_src_dir(start: Path) -> Path:
    for p in [start, *start.parents]:
        candidate = p / "src"
        if (candidate / "discrete_diffusion").exists():
            return candidate
    raise RuntimeError("Could not find repo src directory")

src_dir = _find_src_dir(Path.cwd().resolve())
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from discrete_diffusion.data.datasets import get_star_graph_dataset


In [7]:
# Star graph dataset API smoke test.
cfg = {
    "data_dir": None,
    "train_file": None,
    "validation_file": None,
}

ds = get_star_graph_dataset(cfg)
print(ds)
print("train columns:", ds["train"].column_names)
print("validation columns:", ds["validation"].column_names)
print("train sample prefix:", ds["train"][0]["prefixes"])
print("train sample completion:", ds["train"][0]["completions"])
print("validation sample prefix:", ds["validation"][0]["prefixes"])
print("validation sample completion:", ds["validation"][0]["completions"])


Reading star graph dataset from /home/miki/github/PDiff/data/star/deg_2_path_2_nodes_50_train_200000.txt:   0%|          | 0/200000 [00:00<?, ?it/s]

Reading star graph dataset from /home/miki/github/PDiff/data/star/deg_2_path_2_nodes_50_train_200000.txt: 100%|██████████| 200000/200000 [00:00<00:00, 3347636.51it/s]
Reading star graph dataset from /home/miki/github/PDiff/data/star/deg_2_path_2_nodes_50_test_20000.txt: 100%|██████████| 20000/20000 [00:00<00:00, 4870584.68it/s]

DatasetDict({
    train: Dataset({
        features: ['prefixes', 'completions'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['prefixes', 'completions'],
        num_rows: 20000
    })
})
train columns: ['prefixes', 'completions']
validation columns: ['prefixes', 'completions']
train sample prefix: 49,33|49,1/49,33=
train sample completion: 49,33
validation sample prefix: 23,22|23,6/23,6=
validation sample completion: 23,6


In [8]:
# Loader-path smoke test: build star_graph via get_dataset (tokenization + masks + padding).
from discrete_diffusion.data.loaders import get_dataset
from discrete_diffusion.data.tokenizers import AsciiCharTokenizer

loader_cfg = {
    "data_dir": None,
    "train_file": None,
    "validation_file": None,
}

tokenizer = AsciiCharTokenizer()
cache_dir = str(Path.cwd() / ".cache" / "star_graph_loader_demo")

train_tok = get_dataset(
    dataset_name="star_graph",
    tokenizer=tokenizer,
    wrap=False,
    mode="train",
    cache_dir=cache_dir,
    insert_eos=True,
    insert_special_tokens=True,
    block_size=128,
    num_proc=1,
    streaming=False,
    dataset_config=loader_cfg,
    tokenize=True,
)

valid_tok = get_dataset(
    dataset_name="star_graph",
    tokenizer=tokenizer,
    wrap=False,
    mode="validation",
    cache_dir=cache_dir,
    insert_eos=True,
    insert_special_tokens=True,
    block_size=128,
    num_proc=1,
    streaming=False,
    dataset_config=loader_cfg,
    tokenize=True,
)

print("train rows:", len(train_tok), "valid rows:", len(valid_tok))
print("train columns:", train_tok.column_names)
# Preview using python format to avoid optional torchvision dependency in torch formatter.
example = train_tok.with_format("python")[0]
print("input_ids:", example["input_ids"])
print("attention_mask:", example["attention_mask"])
print("loss_mask:", example["loss_mask"])
print("acc_mask:", example["accuracy_mask"])
print("noise_mask:", example["noise_mask"])

for tok, tid in zip(tokenizer.all_special_tokens, tokenizer.all_special_ids):
    print(f"{tok}: {tid}")


Reading star graph dataset from /home/miki/github/PDiff/data/star/deg_2_path_2_nodes_50_train_200000.txt: 100%|██████████| 200000/200000 [00:00<00:00, 4479275.51it/s]
Reading star graph dataset from /home/miki/github/PDiff/data/star/deg_2_path_2_nodes_50_test_20000.txt: 100%|██████████| 20000/20000 [00:00<00:00, 4654648.76it/s]


Tokenizing:   0%|          | 0/200000 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/200000 [00:00<?, ? examples/s]

Reading star graph dataset from /home/miki/github/PDiff/data/star/deg_2_path_2_nodes_50_train_200000.txt: 100%|██████████| 200000/200000 [00:00<00:00, 4510853.12it/s]
Reading star graph dataset from /home/miki/github/PDiff/data/star/deg_2_path_2_nodes_50_test_20000.txt: 100%|██████████| 20000/20000 [00:00<00:00, 5985449.88it/s]


Tokenizing:   0%|          | 0/20000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

train rows: 200000 valid rows: 20000
train columns: ['prefixes', 'completions', 'input_ids', 'attention_mask', 'loss_mask', 'accuracy_mask', 'noise_mask']
input_ids: [2, 38, 43, 55, 37, 37, 73, 38, 43, 55, 35, 58, 38, 43, 55, 37, 37, 62, 38, 43, 55, 37, 37, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loss_mask: [0, 0, 0, 0, 0,

In [ ]:
# Tiny visualization: token-id cells with per-mask color bands.
from pathlib import Path
import sys

def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "discrete_diffusion").exists():
            return p
    raise RuntimeError("Could not find repo root (src/discrete_diffusion)")

_repo_root = _find_repo_root(Path.cwd().resolve())
_notebook_utils_dir = _repo_root / "notebooks" / "data_vis"
if str(_notebook_utils_dir) not in sys.path:
    sys.path.insert(0, str(_notebook_utils_dir))

from row_vis_utils import plot_token_masks_row

fig, ax = plot_token_masks_row(example, tokenizer=tokenizer, label_mode="chars", max_tokens=64)



In [10]:
# Tokenizer metadata used by star_graph config.
from discrete_diffusion.data.tokenizers import AsciiCharTokenizer

tokenizer = AsciiCharTokenizer()
print("len:", len(tokenizer))
print(
    "bos/eos/pad/mask:",
    tokenizer.bos_token_id,
    tokenizer.eos_token_id,
    tokenizer.pad_token_id,
    tokenizer.mask_token_id,
)

tokenizer.__dict__


len: 77
bos/eos/pad/mask: 2 3 5 4


/home/miki/github/PDiff/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{'characters': ['a',
  'b',
  'c',
  'd',
  'e',
  'f',
  'g',
  'h',
  'i',
  'j',
  'k',
  'l',
  'm',
  'n',
  'o',
  'p',
  'q',
  'r',
  's',
  't',
  'u',
  'v',
  'w',
  'x',
  'y',
  'z',
  '0',
  '1',
  '2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  '!',
  '"',
  '#',
  '$',
  '%',
  '&',
  "'",
  '(',
  ')',
  '*',
  '+',
  ',',
  '-',
  '.',
  '/',
  ':',
  ';',
  '<',
  '=',
  '>',
  '?',
  '@',
  '[',
  '\\',
  ']',
  '^',
  '_',
  '`',
  '{',
  '|',
  '}',
  '~',
  ' '],
 '_vocab_str_to_int': {'[CLS]': 0,
  '[SEP]': 1,
  '[BOS]': 2,
  '[EOS]': 3,
  '[MASK]': 4,
  '[PAD]': 5,
  '[RESERVED]': 6,
  '[UNK]': 7,
  'a': 8,
  'b': 9,
  'c': 10,
  'd': 11,
  'e': 12,
  'f': 13,
  'g': 14,
  'h': 15,
  'i': 16,
  'j': 17,
  'k': 18,
  'l': 19,
  'm': 20,
  'n': 21,
  'o': 22,
  'p': 23,
  'q': 24,
  'r': 25,
  's': 26,
  't': 27,
  'u': 28,
  'v': 29,
  'w': 30,
  'x': 31,
  'y': 32,
  'z': 33,
  '0': 34,
  '1': 35,
  '2': 36,
  '3': 37,
  '4': 38,
  '5': 39,
  '6': 40,
 